BRONZE

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

caminho_clientes = "/Volumes/workspace/default/data_projeto_loja/clientes.csv"
caminho_estoque = "/Volumes/workspace/default/data_projeto_loja/estoque.csv"
caminho_vendas = "/Volumes/workspace/default/data_projeto_loja/vendas.csv"

schemas = {
  "estoque": StructType([
    StructField("produto_id", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("custo_producao", StringType(), True),
    StructField("preco_venda", StringType(), True),
    StructField("quantidade_disponivel", StringType(), True),
    StructField("tempo_producao_horas", StringType(), True), 
    StructField("data_cadastro", StringType(), True),
    StructField("corrupt_record", StringType(), True)]),
  "vendas": StructType([
    StructField("venda_id", StringType(), True),
    StructField("cliente_id", StringType(), True),
    StructField("produto_id", StringType(), True),
    StructField("quantidade", StringType(), True),
    StructField("preco_unitario", StringType(), True),
    StructField("desconto_percent", StringType(), True),
    StructField("forma_pagamento", StringType(), True),
    StructField("data_venda", StringType(), True), 
    StructField("status_entrega", StringType(), True),
    StructField("despesa_id", StringType(), True),
    StructField("tipo_despesa", StringType(), True),
    StructField("valor_despesa", StringType(), True),
    StructField("data_despesa", StringType(), True),
    StructField("corrupt_record", StringType(), True)]),
  "clientes": StructType([
    StructField("cliente_id", StringType(), True),
    StructField("nome_cliente", StringType(), True),
    StructField("email", StringType(), True),
    StructField("telefone", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("data_cadastro", StringType(), True),
    StructField("canal_aquisicao", StringType(), True),
    StructField("corrupt_record", StringType(), True)]),
}

df_clientes = spark.read\
    .format("csv")\
    .schema(schemas["clientes"])\
    .option("header", "true")\
    .option("mode", "PERMISSIVE")\
    .option("multiLine", "true")\
    .option("columnNameOfCorruptRecord", "corrupt_record")\
    .csv(caminho_clientes)

df_estoque = spark.read\
  .format("csv")\
  .schema(schemas["estoque"])\
  .option("header", "true")\
  .option("mode", "PERMISSIVE")\
  .option("multiLine", "true")\
  .option("columnNameOfCorruptRecord", "corrupt_record")\
  .csv(caminho_estoque)

df_vendas = spark.read\
  .format("csv")\
  .schema(schemas["vendas"])\
  .option("header", "true")\
  .option("mode", "PERMISSIVE")\
  .option("multiLine", "true")\
  .option("columnNameOfCorruptRecord", "corrupt_record")\
  .csv(caminho_vendas)

data_ingestao = current_timestamp()
df_clientes = df_clientes.withColumn("data_ingestao", current_timestamp())
df_estoque = df_estoque.withColumn("data_ingestao", current_timestamp())
df_vendas = df_vendas.withColumn("data_ingestao", current_timestamp())

df_clientes.write \
    .option("mergeSchema", "true")\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.default.clientes_bronze")

df_estoque.write \
    .option("mergeSchema", "true")\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.default.estoque_bronze")

df_vendas.write \
    .option("mergeSchema", "true")\
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.default.vendas_bronze")

df_clientes.printSchema() 
df_estoque.printSchema() 
df_vendas.printSchema()
display(df_clientes.limit(5))
display(df_estoque.limit(5))
display(df_vendas.limit(5))
